# KVCacheX on Google Colab: CPU checks and CUDA storage validation

KVCacheX is a **KV-cache memory manager** for LLM inference: block allocation, prefix sharing and eviction,
plus storage backends for the KV values. It has **no attention or model kernels**. This notebook does
**not** run model inference and measures nothing about inference speed.

What this notebook does, in order:

1. Checks for a GPU runtime and records the device, driver and CUDA versions.
2. Installs the Rust toolchain pinned by the project (`rust-toolchain.toml`).
3. Gets the source: **upload the ZIP** made by `./scripts/package_colab.sh` (primary path), or optionally clone a
   Git remote that has content.
4. Runs the CPU checks (format, Clippy, tests, collision-mode tests, simulator invariants).
5. Runs the GPU checks: CUDA device-storage tests (write/read integrity, device-to-device block copy and COW,
   memory lifecycle, randomized model check) and a storage-transfer microbenchmark.
6. Writes a run record (`results/gpu/colab_run.json`) with every command, exit code and test count.

Run the cells top to bottom. Nothing runs in the background, nothing is mounted, and no secrets are needed.
**GPU checks fail loudly when no GPU is attached; they never skip.**

## 0. Helpers

Defines `sh()`, which runs a shell command, keeps the full log under `/content/kvx_logs/`, prints the tail,
parses Rust test counts and records the result in `RUN`.

In [ ]:
import datetime, hashlib, json, os, pathlib, platform, re, shutil, subprocess, sys, time, zipfile

IN_COLAB = "google.colab" in sys.modules
WORK = pathlib.Path("/content") if IN_COLAB else pathlib.Path.cwd()
SRC = WORK / "kvcachex"
LOGS = WORK / "kvx_logs"
LOGS.mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = f"{pathlib.Path.home() / '.cargo' / 'bin'}:{os.environ['PATH']}"
os.environ["CARGO_TERM_COLOR"] = "never"

RUN = {
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "in_colab": IN_COLAB,
    "python": platform.python_version(),
    "os": platform.platform(),
    "env": {},
    "checks": [],
}

TEST_RE = re.compile(r"test result: (ok|FAILED)\. (\d+) passed; (\d+) failed; (\d+) ignored")


def test_counts(out):
    rows = TEST_RE.findall(out)
    return {
        "suites": len(rows),
        "passed": sum(int(r[1]) for r in rows),
        "failed": sum(int(r[2]) for r in rows),
        "ignored": sum(int(r[3]) for r in rows),
    }


def sh(cmd, name=None, cwd=None, check=True, tail=25):
    """Run `cmd` in a shell, log it, record it in RUN, and raise on failure if `check`."""
    name = name or cmd
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    secs = round(time.time() - t0, 1)
    log = LOGS / (re.sub(r"[^A-Za-z0-9_.-]+", "_", name)[:60] + ".log")
    log.write_text(f"$ {cmd}\n{p.stdout}\n[exit {p.returncode}]\n")
    lines = p.stdout.rstrip().splitlines()
    print(f"$ {cmd}")
    print("\n".join(lines[-tail:]) if tail else "")
    print(f"[{name}] exit {p.returncode} in {secs}s (full log: {log})\n")
    rec = {"name": name, "cmd": cmd, "cwd": str(cwd) if cwd else None, "rc": p.returncode, "seconds": secs,
           "tests": test_counts(p.stdout), "log": str(log)}
    RUN["checks"].append(rec)
    if check and p.returncode != 0:
        raise RuntimeError(f"{name} FAILED (exit {p.returncode}); see {log}")
    return p.stdout


print("Colab:", IN_COLAB, "| work dir:", WORK)

## 1. GPU runtime

To use a GPU: **Runtime → Change runtime type → Hardware accelerator: GPU** (for example T4), save, then run the
notebook from the top. If Colab reports that no GPU is available (quota or capacity), the CPU checks below still
run, and the GPU cell will fail with that reason recorded. That is the honest result.

In [ ]:
gpu = {"nvidia_smi": shutil.which("nvidia-smi") is not None}
if gpu["nvidia_smi"]:
    full = sh("nvidia-smi", "nvidia-smi", check=False, tail=20)
    m = re.search(r"CUDA Version:\s*([\d.]+)", full)
    gpu["driver_cuda_version"] = m.group(1) if m else None
    q = sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total,compute_cap --format=csv,noheader",
           "nvidia-smi query", check=False, tail=8)
    if "compute_cap" in q or "Field" in q:  # older drivers lack compute_cap
        q = sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader",
               "nvidia-smi query (no compute_cap)", check=False, tail=8)
    gpu["devices"] = [line.strip() for line in q.strip().splitlines() if line.strip() and "," in line]
# libcuda may be on LD_LIBRARY_PATH (Colab: /usr/lib64-nvidia) rather than in the ldconfig cache.
gpu["libcuda"] = sh("(ldconfig -p | grep -oE '/[^ ]*libcuda\\.so[^ ]*'; for d in $(echo \"$LD_LIBRARY_PATH\" | tr : ' ') /usr/lib64-nvidia; do ls -1 \"$d\"/libcuda.so* 2>/dev/null; done) | sort -u", "libcuda", check=False, tail=5).strip().splitlines()
nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if os.path.exists("/usr/local/cuda/bin/nvcc") else None)
if nvcc:
    out = sh(f"{nvcc} --version", "nvcc", check=False, tail=3)
    m = re.search(r"release ([\d.]+)", out)
    gpu["nvcc_toolkit_version"] = m.group(1) if m else None
GPU_AVAILABLE = bool(gpu.get("devices")) and bool(gpu["libcuda"])
gpu["available"] = GPU_AVAILABLE
RUN["env"]["gpu"] = gpu
print(json.dumps(gpu, indent=2))
print("\nGPU AVAILABLE" if GPU_AVAILABLE else
      "\nNO GPU VISIBLE. CPU checks can run; the GPU cell will FAIL (not skip). "
      "Change the runtime type to GPU and restart if you want GPU validation.")

## 2. Rust toolchain

Installs `rustup` with the official installer (no default toolchain). The exact toolchain is installed in
step 4 from the project's `rust-toolchain.toml`.

In [ ]:
if shutil.which("rustup") is None:
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs -o /tmp/rustup-init.sh && "
       "sh /tmp/rustup-init.sh -y --profile minimal --default-toolchain none", "install rustup", tail=5)
sh("rustup --version", "rustup version", tail=3)

## 3. Get the source

**Primary path: upload a ZIP.** On your machine, in the project checkout, run:

```bash
./scripts/package_colab.sh        # writes dist/kvcachex-src.zip and dist/kvcachex-src.zip.sha256
```

Run the next cell and pick `kvcachex-src.zip` in the upload dialog. Compare the printed SHA-256 with the
`.sha256` file. The archive excludes `.git`, `target/`, `.cargo-home/`, editor/agent config and secret-like files.

In [ ]:
def safe_extract(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        bad = [n for n in names if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts
               or not n.startswith("kvcachex/")]
        if bad:
            raise RuntimeError(f"refusing archive with unexpected paths: {bad[:5]}")
        z.extractall(dest)
    return len(names)

if (SRC / "Cargo.toml").exists():
    print(f"Source already present at {SRC}; delete it to upload a new archive.")
else:
    if not IN_COLAB:
        raise RuntimeError("Upload only works in Colab; place the source at ./kvcachex manually.")
    from google.colab import files
    uploaded = files.upload()  # choose dist/kvcachex-src.zip
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    name, data = next(iter(uploaded.items()))
    zip_path = WORK / name
    zip_path.write_bytes(data)
    digest = hashlib.sha256(data).hexdigest()
    n = safe_extract(zip_path, WORK)
    RUN["source"] = {"method": "zip-upload", "file": name, "bytes": len(data), "sha256": digest, "entries": n}
    print(f"Extracted {n} entries from {name}; sha256 {digest}")
assert (SRC / "Cargo.toml").exists(), "project source not found"
RUN.setdefault("source", {"method": "pre-existing", "path": str(SRC)})

### 3b. Optional: clone instead of uploading

Only if the project has been pushed to a Git remote **with commits**. Leave `REPO_URL` empty to skip. An empty
repository is detected and reported; the cell does not clone it.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<owner>/kvcachex.git" once the remote has content

if REPO_URL and not (SRC / "Cargo.toml").exists():
    refs = sh(f"git ls-remote --heads {REPO_URL}", "git ls-remote", check=False, tail=5).strip()
    if not refs:
        print("The remote has no branches (empty repository). Use the ZIP upload above.")
    else:
        sh(f"git clone --depth 1 {REPO_URL} {SRC}", "git clone", tail=5)
        RUN["source"] = {"method": "git-clone", "url": REPO_URL,
                         "commit": sh("git rev-parse HEAD", "git rev-parse", cwd=SRC, tail=1).strip()}
elif REPO_URL:
    print("Source already present; not cloning.")

## 4. Install the pinned toolchain and record versions

In [ ]:
toolchain = (SRC / "rust-toolchain.toml").read_text()
print(toolchain)
sh("rustup toolchain install || rustup show", "install pinned toolchain", cwd=SRC, tail=8)
RUN["env"]["rustc"] = sh("rustc --version", "rustc", cwd=SRC, tail=1).strip()
RUN["env"]["cargo"] = sh("cargo --version", "cargo", cwd=SRC, tail=1).strip()
RUN["env"]["rust_toolchain_toml"] = toolchain

## 5. CPU checks

These are the same checks the repository's CI runs. They need no GPU. `--all-features` Clippy also
*compiles* the CUDA backend: it loads `libcuda` only at run time, so compiling needs no CUDA toolkit.

In [ ]:
CPU_CHECKS = [
    ("fmt", "cargo fmt --all -- --check"),
    ("clippy", "cargo clippy --locked --all-targets --all-features -- -D warnings"),
    ("cpu-tests", "cargo test --locked --workspace --release"),
    ("collision-tests", "cargo test --locked -p kvcachex --release --features degenerate-hash"),
    ("simulator-invariants",
     "cargo run --locked --release -p kvx -- simulate --workload shared-prefix --requests 100 --check-every 10"),
]
for name, cmd in CPU_CHECKS:
    sh(cmd, name, cwd=SRC, tail=12)
print("CPU checks passed.")

## 6. GPU checks (CUDA device storage)

* `tests/gpu.rs` (feature `cuda`) runs on the device. It checks:
  * write and read integrity of every slot;
  * device-to-device block copies in both directions;
  * error handling for bad arguments and oversized allocations;
  * device memory being returned on drop;
  * COW through `PagedKvCache`;
  * a randomized model check that downloads and verifies every computed KV value after every operation.
* `kvx gpu-bench` times **storage operations only**: host→device writes, device→host reads and device→device block
  copies, each including a stream synchronize, with the CPU backend as a reference. These are transfer costs, not
  inference latency.

This cell **raises** if no GPU is visible or if any GPU test fails or is missing.

In [ ]:
# Reset any verdict from an earlier attempt in this session; only this cell's outcome counts.
RUN.pop("gpu_blocker", None)
RUN["gpu_validated"] = False
if not GPU_AVAILABLE:
    RUN["gpu_validated"] = False
    RUN["gpu_blocker"] = "no GPU visible to this runtime (see section 1)"
    raise RuntimeError("GPU validation NOT run: no GPU visible. Change runtime type to GPU, or record this as a blocker.")

out = sh("cargo test --locked -p kvcachex --release --features cuda --test gpu -- --test-threads=1 --nocapture",
         "gpu-tests", cwd=SRC, tail=30)
m = re.search(r"running (\d+) tests", out)
expected = int(m.group(1)) if m else -1
counts = test_counts(out)
if not (expected > 0 and counts["failed"] == 0 and counts["passed"] == expected and counts["ignored"] == 0):
    raise RuntimeError(f"GPU tests incomplete: expected {expected}, got {counts}")
sh("cargo run --locked --release -p kvx --features cuda -- gpu-info", "gpu-info", cwd=SRC, tail=12)
sh("cargo run --locked --release -p kvx --features cuda -- gpu-bench --json results/gpu/gpu_bench.json",
   "gpu-bench", cwd=SRC, tail=16)
RUN["gpu_validated"] = True
print(f"GPU checks passed: {counts['passed']}/{expected} device tests, benchmark written.")

## 7. Run record

Writes `results/gpu/colab_run.json` (environment, every command, exit codes, test counts) and
`results/gpu/colab_run.md`. To bring them back, use the download cell, then copy `results/gpu/` into your
checkout.

In [ ]:
RUN["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
RUN.setdefault("gpu_validated", False)
out_dir = SRC / "results" / "gpu"
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "colab_run.json").write_text(json.dumps(RUN, indent=2) + "\n")
rows = ["| check | exit | seconds | tests passed/failed/ignored |", "|---|---:|---:|---|"]
for c in RUN["checks"]:
    t = c["tests"]
    tests = f"{t['passed']}/{t['failed']}/{t['ignored']} in {t['suites']} suites" if t["suites"] else ""
    rows.append(f"| {c['name']} | {c['rc']} | {c['seconds']} | {tests} |")
gpu = RUN["env"].get("gpu", {})
md_text = "\n".join([
    "# KVCacheX Colab run", "",
    f"- started {RUN['started_utc']}, finished {RUN['finished_utc']}",
    f"- GPU: {gpu.get('devices')} | driver CUDA {gpu.get('driver_cuda_version')} | nvcc {gpu.get('nvcc_toolkit_version')}",
    f"- {RUN['env'].get('rustc')} | source: {RUN.get('source')}",
    f"- **GPU validated: {RUN['gpu_validated']}**" + (f" (blocker: {RUN.get('gpu_blocker')})" if RUN.get("gpu_blocker") else ""),
    "", *rows, "",
])
(out_dir / "colab_run.md").write_text(md_text)
print(md_text)

In [ ]:
# Optional: download the results (asks the browser to save a file).
if IN_COLAB:
    from google.colab import files
    archive = shutil.make_archive(str(WORK / "kvcachex-colab-results"), "zip", SRC, "results/gpu")
    files.download(archive)